<a href="https://colab.research.google.com/github/tazar09/un_inputs/blob/main/Nairobi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
"""
TAX PROTOCOL ANALYZER – INDIVIDUAL + GROUPED
Outputs (individual, by country):
  1) country_summary.csv
  2) country_detailed_positions.csv
  3) tax_protocol_analysis.csv
  4) country_tax_analysis_report.pdf       (country-by-country)

Additional grouped outputs (Africa / OECD+ / EU / Others):
  5) group_summary.csv
  6) group_tax_analysis_report.pdf         (group overview + countries inside)
  7) country_topic_heatmap.png             (countries x topics)
  8) country_topic_clusters.png            (clustered countries x topics)
  9) group_topic_heatmap.png               (groups x topics)
 10) group_top_topics.png                  (top topic per group)
"""

# ============================================================================
# 1. INSTALL & IMPORTS
# ============================================================================
!pip install matplotlib seaborn pandas numpy fpdf PyPDF2 -q

import re
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
from google.colab import files
import PyPDF2
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# 2. COUNTRY GROUPS
# ============================================================================
COUNTRY_GROUPS = {
    'Africa': [
        'African Union', 'Nigeria', 'Sierra Leone', 'South Africa', 'Egypt', 'Kenya',
        'Ethiopia', 'Ghana', 'Morocco', 'Algeria', 'Angola', 'Benin', 'Botswana',
        'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cameroon', 'Central African Republic',
        'Chad', 'Comoros', 'Congo', 'Djibouti', 'Equatorial Guinea',
        'Eritrea', 'Eswatini', 'Gabon', 'Gambia', 'Guinea', 'Guinea Bissau', 'Lesotho',
        'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius',
        'Mozambique', 'Namibia', 'Niger', 'Rwanda', 'Sao Tome Principe', 'Senegal',
        'Seychelles', 'Somalia', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda',
        'Zambia', 'Zimbabwe', 'Republic of Tanzania', 'Siera -Leone', 'Cote d ivoire'
    ],
    'OECD+': [
        'United States', 'United Kingdom', 'Germany', 'France', 'Japan', 'Italy', 'Canada',
        'Australia', 'Spain', 'South Korea', 'Netherlands', 'Switzerland', 'Turkey',
        'Sweden', 'Norway', 'Belgium', 'Austria', 'Denmark', 'Poland', 'Finland',
        'Ireland', 'New Zealand', 'Czech Republic', 'Portugal', 'Greece', 'Hungary',
        'Slovak Republic', 'Luxembourg', 'Iceland', 'Chile', 'Israel', 'Slovenia',
        'Estonia', 'Latvia', 'Lithuania', 'Colombia', 'Costa Rica', 'Mexico', 'Singapore'

    ],
    'Asia_Latin': ['Russian Federation', 'India', 'China', 'United arab emirates', 'Saudi arabia',
               'Brazil', 'Chile', 'Bangladesh', 'Iran', 'Russian f ederation'
        ],

    'EU': [
        'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic',
        'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary',
        'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands',
        'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden'
    ]
}

def assign_group(country: str) -> str:
    """Assign to Africa, OECD+, EU, or Others, based on exact country name."""
    c_low = country.lower()
    for group, clist in [('Africa', COUNTRY_GROUPS['Africa']),
                         ('OECD+', COUNTRY_GROUPS['OECD+']),
                         ('EU', COUNTRY_GROUPS['EU']),
                         ('Asia_Latin', COUNTRY_GROUPS['Asia_Latin'])]:
        for c in clist:
            if c_low == c.lower():
                return group
    return 'Others'

# ============================================================================
# 3. TOPIC TAGS
# ============================================================================
TAGS = {
    'Transparency': ['simultaneous', 'assistance', 'exchange',
                      'mutual', 'tax examination', 'cooperate',
                      'cooperation', 'tax debts', 'transparency',
                      'information sharing', 'registries', 'registry'
                      'asymmetries', 'asymmetry', 'beneficial ownership'],

    'Overlap': ['overlap', 'overlapping', 'existing', 'duplication',
                'framework'],

    'MAP': ['map', 'mutual agreement procedure', 'mutual agreement'],

    'Arbitration': ['arbitration', 'arbitrator', 'binding arbitration'],
    'Sovereignty': ['sovereignty', 'sovereign', 'domestic law', 'domestic'],

    'Confidentiality': ['guardrails', 'personal data', 'protection', 'safeguards'],

    'Capacity Building': ['capacity-building','capacity building',
                          'technical assistance',
                          'database', 'comparables', 'transfer pricing data',
                          'gap'],

    'Best Practices': ['best practices', 'guidance', 'guidelines'],

    'Optionality': ['optional', 'opt-in', 'opt-out', 'voluntary'],
    'Digital Taxation': ['presence','digital', 'digital tax',
                         'digital services tax', 'dst'],

    'BEPS': ['beps', 'base erosion', 'shifting', 'resource mobilisation',
             'erode', 'aggresive', 'residence'],

    'Transfer pricing': ['transfer pricing', 'arm', 'arms', "arm's",
                         'benchmarking', 'benchmark', 'databases', 'database',
                         'comparables', 'software'],

    'Carbon Tax': ['carbon', 'climate', 'environmental'],
    'Crypto Tax': ['crypto', 'cryptocurrency', 'digital assets', 'blockchain']
}

# ============================================================================
# 4. PDF TEXT EXTRACTION
# ============================================================================
def upload_extract_pdf() -> str:
    print("📁 Upload your PDF...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    filename = list(uploaded.keys())[0]

    text = ""
    with open(filename, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    # Normalize basic issues
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[^\w\s:\.\,\!\?\-\(\)\[\]]', ' ', text)
    text = re.sub(r'[\u2022\u2026\u2013\u2014\u2018\u2019\u201c\u201d]', '-', text)
    return text.strip()

def normalize_country_name(name: str) -> str:
    n = re.sub(r'[^\w\s]', '', name.lower().strip())
    if 'russian' in n and 'feder' in n:
        return "Russian Federation"
    mapping = {
        'african union': 'African Union',
        'united kingdom': 'United Kingdom',
        'uk': 'United Kingdom'
    }
    return mapping.get(n, name.strip())

# ============================================================================
# 5. PARSER (INDIVIDUAL + GROUP METADATA)
# ============================================================================
class TaxParser:
    def __init__(self, text: str):
        self.text = text
        self.countries_raw = {}

    def clean_for_pdf(self, text: str) -> str:
        text = re.sub(r'[\u2022\u2026\u2013\u2014\u2018\u2019\u201c\u201d]', '-', text)
        return text.encode('latin-1', errors='ignore').decode('latin-1')

    def parse_countries(self):
        lines = self.text.split('\n')
        current_country = None
        buffer = []

        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue

            # Country heading: short line, ends with colon
            if stripped.endswith(':') and 2 < len(stripped) < 60:
                # Save previous
                if current_country and buffer:
                    norm = normalize_country_name(current_country)
                    full_raw = " ".join(buffer)
                    self.countries_raw[norm] = buffer[:]
                current_country = stripped[:-1].strip()
                buffer = []
            else:
                if current_country:
                    buffer.append(stripped)

        # Last one
        if current_country and buffer:
            norm = normalize_country_name(current_country)
            self.countries_raw[norm] = buffer[:]

    def count_tags(self, text: str) -> dict:
        t = text.lower()
        out = {}
        for tag, kws in TAGS.items():
            count = 0
            for kw in kws:
                pattern = r'\b' + re.escape(kw.lower()) + r'\b'
                count += len(re.findall(pattern, t))
            out[tag] = count
        return out

    def analyze(self) -> dict:
        self.parse_countries()
        results = {}
        for country, statements in self.countries_raw.items():
            full_raw = " ".join(statements)
            clean_text = self.clean_for_pdf(full_raw)
            positions = self.count_tags(full_raw)
            group = assign_group(country)
            results[country] = {
                'group': group,
                'statements': statements[:5],
                'full_text': clean_text,
                'positions': positions,
                'word_count': len(full_raw.split()),
                'total_mentions': sum(positions.values()),
                'top_topic': max(positions, key=positions.get) if positions else 'None'
            }
        return results

# ============================================================================
# 6. INDIVIDUAL PDF REPORT (COUNTRY-BY-COUNTRY)
# ============================================================================
class CountryPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 8, 'Tax Protocol Position Analysis (By Country)', 0, 1, 'C')
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

    def section_title(self, title):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(52, 152, 219)
        self.set_text_color(255, 255, 255)
        self.cell(0, 7, title, 0, 1, 'L', True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def write_text(self, text):
        text = re.sub(r'[\u2022\u2026\u2013\u2014\u2018\u2019\u201c\u201d]', '-', text)
        text = text.encode('latin-1', errors='ignore').decode('latin-1')
        self.set_font('Arial', '', 10)
        self.multi_cell(0, 5, text)
        self.ln(2)

def generate_country_pdf(country_data: dict, filename: str = "country_tax_analysis_report.pdf"):
    pdf = CountryPDF()
    pdf.add_page()

    # Simple global summary
    pdf.section_title("Summary")
    pdf.write_text(f"Total countries/organizations analyzed: {len(country_data)}")

    # Country-by-country
    pdf.add_page()
    pdf.section_title("Country-by-Country Analysis")
    for country, data in sorted(country_data.items()):
        pdf.set_font('Arial', 'B', 11)
        pdf.cell(0, 6, f"{country} ({data['group']})", 0, 1)
        pdf.set_font('Arial', '', 10)
        pdf.cell(0, 5,
                 f"Words: {data['word_count']} | Mentions: {data['total_mentions']} | Top topic: {data['top_topic']}",
                 0, 1)
        top_topics = sorted(data['positions'].items(), key=lambda x: x[1], reverse=True)[:5]
        if top_topics:
            pdf.cell(0, 5, "Top topics:", 0, 1)
            for tag, count in top_topics:
                pdf.cell(0, 5, f"  - {tag}: {count}", 0, 1)
        pdf.ln(3)

    pdf.output(filename)
    print(f"✅ Country PDF saved: {filename}")

# ============================================================================
# 7. GROUP PDF REPORT (AGGREGATED BY GROUP)
# ============================================================================
class GroupPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 8, 'Tax Protocol Position Analysis (By Group)', 0, 1, 'C')
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

    def section_title(self, title):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(39, 174, 96)
        self.set_text_color(255, 255, 255)
        self.cell(0, 7, title, 0, 1, 'L', True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def write_text(self, text):
        text = re.sub(r'[\u2022\u2026\u2013\u2014\u2018\u2019\u201c\u201d]', '-', text)
        text = text.encode('latin-1', errors='ignore').decode('latin-1')
        self.set_font('Arial', '', 10)
        self.multi_cell(0, 5, text)
        self.ln(2)

def generate_group_pdf(country_data: dict, filename: str = "group_tax_analysis_report.pdf"):
    # Aggregate by group
    group_stats = defaultdict(lambda: {'countries': [], 'words': 0, 'mentions': 0, 'topics': defaultdict(int)})
    for country, data in country_data.items():
        g = data['group']
        group_stats[g]['countries'].append(country)
        group_stats[g]['words'] += data['word_count']
        group_stats[g]['mentions'] += data['total_mentions']
        for tag, count in data['positions'].items():
            group_stats[g]['topics'][tag] += count

    pdf = GroupPDF()
    pdf.add_page()

    # Executive summary
    pdf.section_title("Executive Summary by Group")
    txt = ""
    for g, stats in group_stats.items():
        txt += (f"{g}: {len(stats['countries'])} countries, "
                f"{stats['words']:,} words, {stats['mentions']} topic mentions\n")
    pdf.write_text(txt)

    # Group details
    for g, stats in group_stats.items():
        pdf.add_page()
        pdf.section_title(f"{g} Group")
        pdf.write_text(f"Countries: {', '.join(sorted(stats['countries']))}")

        # Top topics in group
        top_topics = sorted(stats['topics'].items(), key=lambda x: x[1], reverse=True)[:5]
        if top_topics:
            pdf.write_text("Most mentioned topics in this group:")
            for tag, count in top_topics:
                pdf.write_text(f"  - {tag}: {count}")

    pdf.output(filename)
    print(f"✅ Group PDF saved: {filename}")

# ============================================================================
# 8. VISUALISATIONS – INDIVIDUAL & GROUP
# ============================================================================
def create_visualisations(country_data: dict):
    tags = list(TAGS.keys())

    # ---------- INDIVIDUAL HEATMAP (countries x topics) ----------
    countries = list(country_data.keys())
    matrix_c = np.array(
        [[country_data[c]['positions'].get(tag, 0) for tag in tags] for c in countries],
        dtype=float
    )
    plt.figure(figsize=(16, max(6, 0.4*len(countries))))
    sns.heatmap(matrix_c, xticklabels=tags, yticklabels=countries,
                annot=True, fmt='.0f', cmap='YlOrRd',
                cbar_kws={'label': 'Mentions'})
    plt.title('Countries vs Topics Heatmap')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('country_topic_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ country_topic_heatmap.png")

    # ---------- INDIVIDUAL CLUSTERS (countries x topics) ----------
    g = sns.clustermap(matrix_c, row_cluster=True, col_cluster=True,
                       yticklabels=countries, xticklabels=tags,
                       cmap='YlOrRd', figsize=(14, 10))
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    g.fig.suptitle('Topic Clusters by Country', y=1.02)
    g.savefig('country_topic_clusters.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ country_topic_clusters.png")

    # ---------- GROUP-LEVEL AGGREGATION ----------
    group_stats = defaultdict(lambda: defaultdict(int))
    for country, data in country_data.items():
        gname = data['group']
        group_stats[gname]['countries'] += 1
        group_stats[gname]['words'] += data['word_count']
        for tag, count in data['positions'].items():
            group_stats[gname][tag] += count

    groups = list(group_stats.keys())

    # group_topic_heatmap.png (groups x topics)
    matrix_g = np.array([[group_stats[g].get(tag, 0) for tag in tags] for g in groups],
                        dtype=float)
    plt.figure(figsize=(12, 4))
    sns.heatmap(matrix_g, xticklabels=tags, yticklabels=groups,
                annot=True, fmt='.0f', cmap='YlOrRd',
                cbar_kws={'label': 'Total mentions'})
    plt.title('Groups vs Topics Heatmap')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('group_topic_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ group_topic_heatmap.png")

    # group_top_topics.png (top topic per group)
    group_top_rows = []
    for gname in groups:
        top_tag = max(tags, key=lambda t: group_stats[gname].get(t, 0))
        group_top_rows.append({
            'Group': gname,
            'Top_Topic': top_tag,
            'Mentions': group_stats[gname][top_tag]
        })
    df_group_top = pd.DataFrame(group_top_rows)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_group_top, x='Mentions', y='Group', hue='Top_Topic')
    plt.title('Top Topics by Group')
    plt.tight_layout()
    plt.savefig('group_top_topics.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ group_top_topics.png")

# ============================================================================
# 9. MAIN PIPELINE
# ============================================================================
def run_full_analysis():
    print("🚀 TAX PROTOCOL ANALYZER – INDIVIDUAL & GROUPED")
    print("="*70)

    text = upload_extract_pdf()
    parser = TaxParser(text)
    country_data = parser.analyze()

    if not country_data:
        print("❌ No countries found. Ensure headings look like 'Country Name:'.")
        return

    print(f"✅ Parsed {len(country_data)} countries")

    # ---------- INDIVIDUAL CSVs ----------
    summary_rows = []
    detailed_rows = []
    for country, data in country_data.items():
        summary_rows.append({
            'Country': country,
            'Group': data['group'],
            'Word_Count': data['word_count'],
            'Top_Topic': data['top_topic'],
            'Total_Mentions': data['total_mentions']
        })
        drow = {'Country': country, 'Group': data['group']}
        drow.update(data['positions'])
        detailed_rows.append(drow)

    df_summary = pd.DataFrame(summary_rows)
    df_detailed = pd.DataFrame(detailed_rows).fillna(0)
    df_summary.to_csv('country_summary.csv', index=False)
    df_detailed.to_csv('country_detailed_positions.csv', index=False)
    df_summary.to_csv('tax_protocol_analysis.csv', index=False)
    print("✅ Individual CSVs: country_summary.csv, country_detailed_positions.csv, tax_protocol_analysis.csv")

    # ---------- GROUP CSV ----------
    group_rows = []
    group_agg = defaultdict(lambda: {'countries': 0, 'words': 0, 'mentions': 0})
    for country, data in country_data.items():
        gname = data['group']
        group_agg[gname]['countries'] += 1
        group_agg[gname]['words'] += data['word_count']
        group_agg[gname]['mentions'] += data['total_mentions']
    for gname, stats in group_agg.items():
        group_rows.append({
            'Group': gname,
            'Country_Count': stats['countries'],
            'Total_Words': stats['words'],
            'Total_Mentions': stats['mentions']
        })
    df_group = pd.DataFrame(group_rows)
    df_group.to_csv('group_summary.csv', index=False)
    print("✅ Group CSV: group_summary.csv")

    # ---------- PDFs ----------
    generate_country_pdf(country_data, "country_tax_analysis_report.pdf")
    generate_group_pdf(country_data, "group_tax_analysis_report.pdf")

    # ---------- Visuals ----------
    create_visualisations(country_data)

    print("\n🎉 DONE. Generated:")
    print("  • country_summary.csv")
    print("  • country_detailed_positions.csv")
    print("  • tax_protocol_analysis.csv")
    print("  • group_summary.csv")
    print("  • country_tax_analysis_report.pdf")
    print("  • group_tax_analysis_report.pdf")
    print("  • country_topic_heatmap.png")
    print("  • country_topic_clusters.png")
    print("  • group_topic_heatmap.png")
    print("  • group_top_topics.png")

# ============================================================================
# 10. RUN
# ============================================================================
run_full_analysis()


🚀 TAX PROTOCOL ANALYZER – INDIVIDUAL & GROUPED
📁 Upload your PDF...


Saving all-together(1).pdf to all-together(1) (1).pdf
✅ Parsed 34 countries
✅ Individual CSVs: country_summary.csv, country_detailed_positions.csv, tax_protocol_analysis.csv
✅ Group CSV: group_summary.csv
✅ Country PDF saved: country_tax_analysis_report.pdf
✅ Group PDF saved: group_tax_analysis_report.pdf
✅ country_topic_heatmap.png
✅ country_topic_clusters.png
✅ group_topic_heatmap.png
✅ group_top_topics.png

🎉 DONE. Generated:
  • country_summary.csv
  • country_detailed_positions.csv
  • tax_protocol_analysis.csv
  • group_summary.csv
  • country_tax_analysis_report.pdf
  • group_tax_analysis_report.pdf
  • country_topic_heatmap.png
  • country_topic_clusters.png
  • group_topic_heatmap.png
  • group_top_topics.png
